# ECEi MC evaluation dataloader

Isolated dataloader for **ecei_mc** decimated H5 data, matching the setup used by
`run_tcn_baseline_160_original_instancenorm.sh` for training. Use this notebook to
evaluate TCN checkpoints on the same data (clear_decimated + disrupt_decimated).

**Data paths (SciServer):**
- Disrupt: `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/disrupt_decimated`
- Clear:   `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/clear_decimated`

## Setup paths and imports

In [ ]:
import sys
from pathlib import Path

# soen_fusion_zero root: prefer cwd if we're in it, else parent, else fallback
ROOT = Path.cwd()
if (ROOT / "train_tcn_ddp_original.py").exists():
    pass
elif (ROOT.parent / "train_tcn_ddp_original.py").exists():
    ROOT = ROOT.parent
else:
    ROOT = Path("/home/idies/workspace/Storage/yhuang2/persistent/soen_fusion_zero")
sys.path.insert(0, str(ROOT))

# ecei_mc decimated H5 folders (same as run_tcn_baseline_160_original_instancenorm.sh)
DECIMATED_ROOT = "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/disrupt_decimated"
CLEAR_DECIMATED_ROOT = "/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/clear_decimated"
NORM_STATS = "/home/idies/workspace/Storage/yhuang2/persistent/ecei/norm_stats.npz"
DISRUPT_FILE = str(ROOT / "disruptcnn" / "shots" / "d3d_disrupt_ecei.final.txt")
CLEAR_FILE = str(ROOT / "disruptcnn" / "shots" / "d3d_clear_ecei.final.txt")

print("ROOT:", ROOT)
print("DECIMATED_ROOT:", DECIMATED_ROOT)
print("CLEAR_DECIMATED_ROOT:", CLEAR_DECIMATED_ROOT)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from disruptcnn.dataset_original import EceiDatasetOriginal, OriginalStyleDatasetForDDP

## Build dataset (same config as 160-original-instancenorm training)

Parameters match `train_tcn_ddp_original.py` defaults used by the run script: flattop_only, data_step=10, nsub=781_250, nrecept from 30k output receptive field.

In [ ]:
DATA_STEP = 10
NSUB = 781_250
NRECEPT_TARGET = 30_000  # output receptive field; dataset uses nrecept_raw = NRECEPT_TARGET * DATA_STEP
NRECEPT_RAW = NRECEPT_TARGET * DATA_STEP

inner_ds = EceiDatasetOriginal(
    root="/home/idies/workspace/Storage/yhuang2/persistent/ecei/dsrpt",  # not used when decimated_root set
    disrupt_file=DISRUPT_FILE,
    clear_file=CLEAR_FILE,
    flattop_only=True,
    normalize=True,
    data_step=DATA_STEP,
    nsub=NSUB,
    nrecept=NRECEPT_RAW,
    decimated_root=DECIMATED_ROOT,
    clear_decimated_root=CLEAR_DECIMATED_ROOT,
    norm_stats_path=NORM_STATS,
    decimate_extra=None,
)
ds = OriginalStyleDatasetForDDP(inner_ds)

print("Total sequences:", len(ds))
print("seq_has_disrupt sum:", ds.seq_has_disrupt.sum())

## Train/val/test splits and eval dataloader

In [ ]:
train_idx = ds.get_split_indices("train")
val_idx = ds.get_split_indices("test")
if len(val_idx) == 0:
    val_idx = ds.get_split_indices("val")

print("Train:", len(train_idx), "Val:", len(val_idx))

val_subset = Subset(ds, val_idx)
eval_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

# One batch for shape check
X, target, weight = next(iter(eval_loader))
print("Batch X shape:", X.shape, "target:", target.shape)

## Optional: load checkpoint and run evaluation

Set `CHECKPOINT_PATH` to a `best.pt` or `epoch_XXXX.pt` from training. The model is built with the same TCN config (levels, nhid, kernel_size, etc.) and instance norm as in the run script.

In [ ]:
CHECKPOINT_PATH = None  # e.g. ROOT / "checkpoints_tcn_ddp_original/L4_H80_YYYYMMDD_HHMMSS/best.pt"

if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    from train_tcn_ddp_original import build_model
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    args = ckpt.get("args", {})
    nrecept_ckpt = ckpt.get("nrecept", NRECEPT_TARGET)
    model, nrecept, _ = build_model(
        args.get("input_channels", 160), 1,
        args.get("levels", 4), args.get("nhid", 80),
        args.get("kernel_size", 15), args.get("dilation_base", 10), args.get("dropout", 0.1),
        nrecept_target=nrecept_ckpt,
        use_instance_norm=args.get("use_instance_norm", True),
        use_prenorm=args.get("use_prenorm", False),
    )
    state = ckpt["state_dict"]
    if next(iter(state.keys())).startswith("module."):
        state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print("Model loaded. nrecept =", nrecept)
else:
    model = None
    nrecept = NRECEPT_TARGET
    print("Set CHECKPOINT_PATH to run model evaluation.")